# Benchmark Service Setup

Run this notebook **once** (or whenever you need to re-index).
It creates the `eval_benchmark` Colette service and builds:
1. Vector embeddings (ChromaDB)
2. BM25 full-text index (Tantivy)
3. hindex hierarchy (topic navigation)

Once done, run `evaluate_retrieval.ipynb` as many times as you like — it connects to the already-running service without re-indexing.


## 1. Configuration


In [ ]:
import os, sys
from pathlib import Path

colette_root  = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(colette_root, "src"))
BENCHMARK_DIR = Path(os.getcwd()) / "benchmark"

# ── Same setup as get_start_python_api.ipynb / app_colette ──────────────────
app_name        = "eval_benchmark"
app_dir         = os.path.join(colette_root, "app_eval_benchmark")
models_repo     = os.path.join(colette_root, "models")
embedding_model = "Alibaba-NLP/gme-Qwen2-VL-2B-Instruct"
llm_model       = "Qwen/Qwen3.5-9B"
llm_gpu_ids     = [1]   # GPU 1 is free; GPU 0 holds embedding, GPU 3 holds external vLLM
embedding_gpu   = 0
TOP_K           = 4
# ────────────────────────────────────────────────────────────────────────────

# Point the indexer directly at the benchmark folder.
# preprocessing.files=["pdf"] in the service config ensures only PDF files are indexed,
# so qa_dataset.jsonl and any other files here are ignored automatically.
PDFS_DIR = BENCHMARK_DIR

from colette.jsonapi import JSONApi
from colette.apidata import APIData

colette_api = JSONApi()
print(f"app_name    : {app_name}")
print(f"app_dir     : {app_dir}")
print(f"models_repo : {models_repo}")
print(f"embedding   : {embedding_model}  (gpu {embedding_gpu})")
print(f"llm         : {llm_model}  (gpu {llm_gpu_ids})")
print(f"pdfs dir    : {PDFS_DIR}  ({len(list(PDFS_DIR.glob('*.pdf')))} PDFs)")


## 2. Create Service


In [ ]:
app_dir = os.path.join(colette_root, "app_eval_benchmark")

service_config = {
    "app": {
        "repository": app_dir,
        "models_repository": models_repo,
        "verbose": "info",
    },
    "parameters": {
        "input": {
            "lib": "hf",
            "preprocessing": {
                "files": ["pdf"],   # only ingest PDF files — ignores qa_dataset.jsonl etc.
            },
            "rag": {
                "embedding_model": embedding_model,
                "gpu_id": embedding_gpu,
                "top_k": TOP_K,
                "retrieval_mode": "hybrid",
                "text_search_engine_top_k": TOP_K,
                "text_search_engine_fields": ["content", "source"],
                "text_search_engine_max_chars_per_doc": 4000,
                "text_search_engine_max_total_chars": 16000,
                "ragm": {
                    "layout_detection": True,
                    "image_width": 512,
                    "image_height": 512,
                    "auto_scale_for_font": False,
                },
            },
            "data": [],
        },
        "llm": {
            "source": llm_model,
            "gpu_ids": llm_gpu_ids,
            "image_width": 320,
            "image_height": 480,
            "adjust_aspect_ratio": True,
            "shared_model": True,
            "inference": {
                "lib": "huggingface",
                "sampling_params": {"temperature": 0},
            },
            "vllm_enforce_eager": True,
            "dtype": "auto",
        },
    },
}

print(f"app_dir      : {app_dir}")
print(f"embedding    : {embedding_model}  (gpu {embedding_gpu})")
print(f"llm          : {llm_model}  (gpu {llm_gpu_ids})")
print(f"documents    : {BENCHMARK_DIR}  (only *.pdf files indexed)")


In [ ]:
# Delete the service if it already exists, then create fresh
try:
    await colette_api.service_delete(app_name)
    print("Deleted existing service.")
except Exception:
    pass  # no-op if service doesn't exist yet

result = colette_api.service_create(app_name, APIData(**service_config))
print("Service created:", result)


## 3. Index Documents


### 2a. Build Hybrid Index (vector embeddings + BM25)

Indexes all PDFs in the benchmark folder. Automatically detects and adds new PDFs without
a full rebuild — only the first run (or after deleting `mm_index/`) triggers a full index.


In [ ]:
mm_index_dir    = Path(app_dir) / "mm_index"
text_search_dir = mm_index_dir / "text_search_engine"
sentinel_file   = mm_index_dir / ".indexed_pdfs.txt"

chroma_ok = mm_index_dir.exists() and any(
    p for p in mm_index_dir.iterdir()
    if p.name not in ("hindex", "text_search_engine", ".indexed_pdfs.txt")
)
bm25_ok = text_search_dir.exists() and any(text_search_dir.iterdir())

# Detect new PDFs that weren't part of the last index run
current_pdfs = {p.name for p in PDFS_DIR.glob("*.pdf")}
indexed_pdfs = set(sentinel_file.read_text().splitlines()) if sentinel_file.exists() else set()
new_pdfs = current_pdfs - indexed_pdfs

if chroma_ok and bm25_ok and not new_pdfs:
    print(f"Hybrid index already up to date ({len(indexed_pdfs)} PDFs) — skipping.")
elif not chroma_ok or not bm25_ok:
    # First run: full build
    missing = []
    if not chroma_ok: missing.append("embeddings")
    if not bm25_ok:   missing.append("BM25/Tantivy")
    print(f"Building hybrid index from scratch (missing: {', '.join(missing)}, {len(current_pdfs)} PDFs)…")
    colette_api.service_index(app_name, APIData(**{
        "parameters": {
            "input": {
                "data": [str(PDFS_DIR)],
                "rag": {"retrieval_mode": "hybrid", "reindex": True},
            }
        }
    }))
    sentinel_file.write_text("\n".join(sorted(current_pdfs)))
    print(f"Hybrid index built ({len(current_pdfs)} PDFs).")
else:
    # Incremental update: add only the new PDFs
    print(f"Adding {len(new_pdfs)} new PDF(s) to hybrid index: {sorted(new_pdfs)}")
    colette_api.service_index(app_name, APIData(**{
        "parameters": {
            "input": {
                "data": [str(PDFS_DIR)],
                "rag": {
                    "retrieval_mode": "hybrid",
                    "update_index": True,
                    "reindex": False,
                },
            }
        }
    }))
    sentinel_file.write_text("\n".join(sorted(current_pdfs)))
    print(f"Hybrid index updated ({len(current_pdfs)} PDFs total).")


### 2b. Build hindex Hierarchy

Adds topic-navigation JSON files on top of the existing vector/BM25 index.
Requires the LLM — expect ~3-5 min per document.


In [ ]:
hindex_dir = Path(app_dir) / "mm_index" / "hindex"
hindex_files = sorted(hindex_dir.glob("*.json")) if hindex_dir.exists() else []

# Count only hindex files that correspond to actual PDFs (ignore any stray entries)
pdf_stems = {Path(f).stem for f in PDFS_DIR.glob("*.pdf")}
valid_hindex = [f for f in hindex_files if any(stem in f.stem for stem in pdf_stems)]
n_pdfs = len(list(PDFS_DIR.glob("*.pdf")))

if len(valid_hindex) >= n_pdfs:
    print(f"hindex already built ({len(valid_hindex)}/{n_pdfs} documents) — skipping.")
else:
    print(f"Building hindex hierarchy ({len(valid_hindex)}/{n_pdfs} documents present)…")
    index_cfg_hindex = {
        "parameters": {
            "input": {
                "data": [str(PDFS_DIR)],
                "preprocessing": {
                    "files": ["pdf"],   # guard: skip qa_dataset.jsonl and any other non-PDF files
                },
                "rag": {
                    "retrieval_mode": "vision_hierarchical",
                    "update_index": True,
                    "reindex": False,
                    "hindex": {
                        "toc_scan_pages": 20,
                        "add_node_summaries": True,
                    },
                },
            }
        }
    }
    colette_api.service_index(app_name, APIData(**index_cfg_hindex))
    print("hindex built.")

# Verify
hindex_files = sorted(hindex_dir.glob("*.json")) if hindex_dir.exists() else []
valid_hindex = [f for f in hindex_files if any(stem in f.stem for stem in pdf_stems)]
print(f"hindex files ({len(valid_hindex)}/{n_pdfs}): {[f.name for f in valid_hindex]}")


## Done

The service is ready. Open `evaluate_retrieval.ipynb` to run the full evaluation.
